# Module 4 - Topic 1: What is Machine Learning
## Live Demo Notebook

**Scenario:** A Nigerian microfinance company wants to automate loan approval decisions.
Their current system uses a hand-written rule: *"Approve if income ≥ ₦200k AND credit score ≥ 600 AND employment ≥ 1 year."*

We'll show two approaches side by side:
1. **Traditional programming** - the rules approach (what they have now)
2. **Machine Learning** - the model learns the rules from historical data

---

## Part 1 - Traditional Programming: Writing Rules by Hand

In [ ]:
# Traditional approach - a human wrote every rule
def approve_loan_rules(monthly_income, credit_score, employment_years):
    """
    Hand-crafted rule: approve if ALL conditions are met.
    A human analyst sat down and defined these thresholds.
    """
    if monthly_income >= 200000 and credit_score >= 600 and employment_years >= 1:
        return 1  # Approved
    return 0      # Rejected

# Test the rule on a few applicants
test_cases = [
    (350000, 720, 5,  "Amaka - senior nurse"),
    (150000, 650, 3,  "Chidi - junior teacher"),
    (500000, 580, 10, "Tunde - long-serving but low credit"),
    (80000,  450, 0,  "Fatima - recent graduate, no history"),
]

print('Rule-Based Loan Decisions:')
print(f'{"Applicant":<35} {"Income":>10} {"Credit":>8} {"Yrs":>5} {"Decision":>10}')
print('-' * 75)
for income, credit, yrs, name in test_cases:
    decision = 'APPROVED' if approve_loan_rules(income, credit, yrs) else 'REJECTED'
    print(f'{name:<35} {income:>10,} {credit:>8} {yrs:>5} {decision:>10}')

In [ ]:
# The problem with rules:
# Tunde has excellent income and 10 years of employment but gets REJECTED
# because his credit score is 580 - just 20 points below the threshold.
# A human loan officer might approve Tunde. The rigid rule doesn't.

# What if income should compensate for a slightly lower credit score?
# What if employment years matter more for older applicants?
# You'd need to write more and more rules - and they'd start conflicting.

print('Problem with rules:')
print('  Tunde: income=₦500,000, employment=10 years, credit=580')
print('  Decision: REJECTED - just 20 points below threshold')
print()
print('A human loan officer might consider:')
print('  - ₦500k income suggests very low default risk')
print('  - 10 years employment = stable')
print('  - Credit score of 580 is borderline, not catastrophic')
print()
print('Rules cannot capture this nuance. ML can.')

---
## Part 2 - Machine Learning: The Model Learns the Rules

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load historical loan data - 20 past decisions
df = pd.read_csv('loan_applicants.csv')
print('Dataset shape:', df.shape)
print()
df.head(8)

In [ ]:
# Separate features (X) from the label (y)
# Features = what we know about the applicant
# Label    = the outcome we want to predict

X = df[['monthly_income', 'credit_score', 'employment_years']]
y = df['loan_approved']

print('Features (X) - 3 columns, 20 rows:')
print(X.head(3))
print()
print('Label (y) - what we want to predict:')
print(y.values)
print(f'Approved: {y.sum()} | Rejected: {(y==0).sum()}')

In [ ]:
# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train)} applicants')
print(f'Test set:     {len(X_test)} applicants')
print()

# Train the ML model - it learns the rules from training data
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)   # <-- this is where learning happens

print('Model trained. It has now seen', len(X_train), 'examples.')
print('It learned patterns - no rules were written by hand.')

In [ ]:
# Evaluate on test set - applicants the model has NEVER seen
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print('Test set predictions vs actual decisions:')
print(f'{"Actual":<10} {"ML Predicted":<15} {"Match?"}')
print('-' * 35)
for actual, predicted in zip(y_test, y_pred):
    match = 'YES' if actual == predicted else 'NO'
    label_a = 'APPROVED' if actual    else 'REJECTED'
    label_p = 'APPROVED' if predicted else 'REJECTED'
    print(f'{label_a:<10} {label_p:<15} {match}')

print()
print(f'Accuracy: {accuracy*100:.0f}%')

---
## Part 3 - Comparison: Rules vs ML on the Same Applicants

In [ ]:
# Compare both approaches on the four test cases from Part 1
import numpy as np

test_applicants = pd.DataFrame({
    'name':             ['Amaka', 'Chidi', 'Tunde', 'Fatima'],
    'monthly_income':   [350000,  150000,  500000,  80000],
    'credit_score':     [720,     650,     580,     450],
    'employment_years': [5,       3,       10,      0],
})

X_new = test_applicants[['monthly_income', 'credit_score', 'employment_years']]

rules_decisions = [
    approve_loan_rules(r.monthly_income, r.credit_score, r.employment_years)
    for _, r in test_applicants.iterrows()
]
ml_decisions = model.predict(X_new)

print(f'{"Applicant":<10} {"Rules":>10} {"ML Model":>10}')
print('-' * 35)
for i, row in test_applicants.iterrows():
    r_dec = 'APPROVED' if rules_decisions[i] else 'REJECTED'
    m_dec = 'APPROVED' if ml_decisions[i]    else 'REJECTED'
    print(f'{row["name"]:<10} {r_dec:>10} {m_dec:>10}')

print()
print('Notice: Tunde (₦500k income, 10 years employment, credit 580)')
print('Rules: REJECTED (credit 580 < 600 threshold)')
print('ML:    may give a different answer based on learned patterns')

---
## Part 4 - What Did the Model Learn?

In [ ]:
import matplotlib.pyplot as plt

# Feature importance - which feature mattered most to the model?
importance = pd.Series(
    model.feature_importances_,
    index=['monthly_income', 'credit_score', 'employment_years']
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importance - What the Model Learned to Pay Attention To',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('The model assigned importance to each feature based on the training data.')
print('This is the pattern it learned - no human specified these weights.')

---
## Summary

| Approach | How it works | Strength | Weakness |
|---|---|---|---|
| **Rule-based** | Human writes every condition | Fully explainable, instant to deploy | Rigid - breaks on edge cases, hard to update |
| **ML** | Model learns patterns from examples | Flexible, handles complex patterns, improves with data | Needs data to train, harder to explain |

**Key takeaway:** ML doesn't replace rules - it learns them automatically from data. When the patterns are too complex for a human to write by hand, ML is the right tool.

**Next - Topic 2:** Types of Machine Learning. We see supervised (like this demo), unsupervised, and reinforcement learning side by side.